In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

In [2]:
from monice import *
from preprocessing import *

In [3]:
def load_and_preprocess_data():
    """Load and preprocess the cirrhosis dataset."""
    print("Loading cirrhosis dataset...")
    
    # Fetch dataset
    cirrhosis = fetch_ucirepo(id=878)
    X = cirrhosis.data.features
    y = cirrhosis.data.targets['Status']
    df = pd.DataFrame(X, columns=cirrhosis.data.feature_names)
    df['Status'] = y

    # Define feature types
    num_feat = ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 
                'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']
    cat_feat = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
    
    # Clean data
    df = df.replace(['NaNN', 'nan', ''], np.nan)
    
    # Fill missing values
    for col in cat_feat:
        if col in df.columns:
            df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    df = df.dropna()
    
    print(f"Dataset shape: {df.shape}")
    print(f"Class distribution:\n{df['Status'].value_counts()}")
    
    # Get feature indices
    feature_names = [col for col in df.columns if col != 'Status']
    cat_idx = [i for i, col in enumerate(feature_names) if col in cat_feat]
    num_idx = [i for i, col in enumerate(feature_names) if col in num_feat]
    
    # Create label encoders for categorical features
    feature_encoders = {}
    df_encoded = df.copy()
    
    for i, col in enumerate(feature_names):
        if i in cat_idx:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df[col])
            feature_encoders[i] = le
    
    return df_encoded, feature_names, cat_idx, num_idx, feature_encoders

In [4]:
def train_model(df, feature_names, num_idx, cat_idx):
    """Train a Random Forest classifier."""
    print("\nTraining model...")
    
    X = df[feature_names].values.astype(np.float64)
    y = df['Status'].values

    # Create preprocessing pipeline
    preprocessor = preprocessing_pipeline(num_idx, cat_idx)

    # Label encoding
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"Label mapping: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}") 
    y_encoded = y_encoded.astype(np.float64)
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # Train model with grid search
    rf_params = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20],
        'min_samples_split': [2, 5]
    }
    
    rf = RandomForestClassifier(random_state=42, class_weight='balanced')
    rf_grid = GridSearchCV(rf, rf_params, cv=3, n_jobs=-1, verbose=0)
    
    clf = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', rf_grid)
    ])

    clf.fit(X_train, y_train)
    
    print(f"Training accuracy: {clf.score(X_train, y_train):.4f}")
    print(f"Test accuracy: {clf.score(X_test, y_test):.4f}")

    # Create prediction function
    predict_fn = lambda x: clf.predict_proba(x)

    return X_train, X_test, y_train, y_test, preprocessor, label_encoder, predict_fn

In [5]:
def train_plausibility_model(X_train, preprocessor):
    """Train an autoencoder for plausibility modeling."""
    print("\nTraining plausibility model...")
    
    grid_params = {
        'learning_rate_init': [0.01, 0.001],
        'alpha': [0.0001, 0.001]
    }
    
    ae_model = MLPAutoencoder(X_train, grid_params, preprocessor=preprocessor)
    ae_model.fit()
    
    return ae_model

In [6]:
# Load data
df, feature_names, cat_idx, num_idx, feature_encoders = load_and_preprocess_data()

# Train model
X_train, X_test, y_train, y_test, preprocessor, label_encoder, predict_fn = \
    train_model(df, feature_names, num_idx, cat_idx)

# Train plausibility model
plausibility_model = train_plausibility_model(X_train, preprocessor)


# Define integer features for MONICE constructor
integer_features = [i for i, name in enumerate(feature_names) 
                    if name in ['Age', 'Copper', 'Alk_Phos', 'Tryglicerides', 'Platelets', 'Stage']]


# Define cost weights (changing age is more expensive)
cost_weights = {
    i: 2.0 for i, name in enumerate(feature_names) if name in ['Age']
}
immutable_features=[i for i, name in enumerate(feature_names) if name in ['Sex']]

monotonic_constraints={
        i: 'decreasing' for i, name in enumerate(feature_names) if name in ['Age']
    }

Loading cirrhosis dataset...
Dataset shape: (276, 18)
Class distribution:
Status
C     147
D     111
CL     18
Name: count, dtype: int64

Training model...
Label mapping: {'C': 0, 'CL': 1, 'D': 2}
Training accuracy: 1.0000
Test accuracy: 0.7679

Training plausibility model...
Autoencoder hidden layers: (9, 5, 4, 5, 9)
Best autoencoder params: {'alpha': 0.0001, 'learning_rate_init': 0.01}


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 276 entries, 0 to 311
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Drug           276 non-null    int32  
 1   Age            276 non-null    int64  
 2   Sex            276 non-null    int32  
 3   Ascites        276 non-null    int32  
 4   Hepatomegaly   276 non-null    int32  
 5   Spiders        276 non-null    int32  
 6   Edema          276 non-null    int32  
 7   Bilirubin      276 non-null    float64
 8   Cholesterol    276 non-null    object 
 9   Albumin        276 non-null    float64
 10  Copper         276 non-null    object 
 11  Alk_Phos       276 non-null    float64
 12  SGOT           276 non-null    float64
 13  Tryglicerides  276 non-null    object 
 14  Platelets      276 non-null    object 
 15  Prothrombin    276 non-null    float64
 16  Stage          276 non-null    float64
 17  Status         276 non-null    object 
dtypes: float64(6), 

In [8]:
# Initialize MONICE
print("\nInitializing MONICE...")
monice = MONICE(
            X_train=X_train,
            y_train=y_train,
            predict_fn=predict_fn,
            plausibility_model=plausibility_model,
            cat_feats=cat_idx,
            num_feats=num_idx,
            integer_feats=integer_features,
            cost_weights=cost_weights,
            immutable_features=immutable_features,
            feature_bounds=None,
            monotonic_constraints=monotonic_constraints,
            distance_metric='gower',
            justified_cf=True,
            eps=1e-6,
            verbose=True)


Initializing MONICE...
Bounds for numerical feature 1: (9598.0, 28018.0)
Bounds for numerical feature 7: (0.3, 28.0)
Bounds for numerical feature 8: (120.0, 1600.0)
Bounds for numerical feature 9: (2.1, 4.4)
Bounds for numerical feature 10: (9.0, 588.0)
Bounds for numerical feature 11: (289.0, 13862.4)
Bounds for numerical feature 12: (28.38, 457.25)
Bounds for numerical feature 13: (33.0, 598.0)
Bounds for numerical feature 14: (62.0, 563.0)
Bounds for numerical feature 15: (9.0, 17.1)
Bounds for numerical feature 16: (1.0, 4.0)
Allowed values for categorical feature 0: [0.0, 1.0]
Allowed values for categorical feature 2: [0.0, 1.0]
Allowed values for categorical feature 3: [0.0, 1.0]
Allowed values for categorical feature 4: [0.0, 1.0]
Allowed values for categorical feature 5: [0.0, 1.0]
Allowed values for categorical feature 6: [0.0, 1.0, 2.0]


In [9]:
y_pred = predict_fn(X_test).argmax(axis=1)
X_explain = X_test[y_pred == y_test]

In [10]:
test_instance = X_explain[10:11]
y_test = predict_fn(test_instance).argmax(axis=1)
label_encoder.inverse_transform(y_test)

array(['D'], dtype=object)

In [11]:
results = monice.explain(
    X=X_explain[11:12],
    target_classes='other',
    optimization=['robustness', 'sparsity', 'proximity', 'plausibility'],
    k_nearest=3,
    n_cfs=3,
    numerical_steps=[0.2, 0.4, 0.6, 0.8, 1.0]
)


MONICE COUNTERFACTUAL EXPLANATION
Optimization objectives: ['robustness', 'sparsity', 'proximity', 'plausibility']
K-nearest neighbors: 3
Number of counterfactuals: 3
Population size: 9
Max generations: 38
Numerical steps: [0.2, 0.4, 0.6, 0.8, 1.0]

Processing target class: 1
------------------------------
Found 14 candidates
Selected 3 valid nearest neighbors

CONSTRAINED MULTI-OBJECTIVE OPTIMIZATION
--------------------------------------------------
Target class: 1
Population size: 9
Max generations: 38
Objectives: ['robustness', 'sparsity', 'proximity', 'plausibility']
--------------------------------------------------
Initial population size: 66
Number of Pareto fronts: 10
Initial population size: 18

Generation 1/38
  Current population size: 18
Generated 100 valid offspring
Number of Pareto fronts: 8

Generation 2/38
  Current population size: 9
Generated 100 valid offspring
Number of Pareto fronts: 7

Generation 3/38
  Current population size: 9
Generated 100 valid offspring
Nu

In [12]:
def display_counterfactuals(results, feature_names, feature_encoders, label_encoder, original_instance_idx=11):

    original = X_explain[original_instance_idx:original_instance_idx+1][0]
    original_class = predict_fn(original.reshape(1, -1)).argmax(axis=1)[0]
    original_class_name = label_encoder.inverse_transform([original_class])[0]
    
    print(f"Original instance predicted as: {original_class_name}")
    
    # Create a DataFrame for the original instance
    original_df = pd.DataFrame([original], columns=feature_names)
    
    # Convert encoded categorical values back to original values
    for feat_idx, encoder in feature_encoders.items():
        if feat_idx < len(feature_names): 
            feat_name = feature_names[feat_idx]
            if feat_name in original_df.columns:
                # Check if the value needs decoding (if it's categorical)
                if feat_idx in feature_encoders:
                    try:
                        original_df[feat_name] = encoder.inverse_transform(original_df[feat_name].astype(int))
                    except:
                        pass
    
    print("\nOriginal instance:")
    display(original_df)
    
    # Process each target class
    for target_class, cf_result in results.items():
        target_class_name = label_encoder.inverse_transform([target_class])[0]
        print(f"\n----- Target Class: {target_class_name} -----")
        
        # Display nearest neighbors first
        print("\nNearest Neighbors (only showing differences from original):")
        nearest_neighbors = cf_result.nearest_neighbors
        
        # Create a DataFrame for the nearest neighbors
        nn_df = pd.DataFrame(nearest_neighbors, columns=feature_names)
        
        # Convert encoded categorical values back to original values for nearest neighbors
        for feat_idx, encoder in feature_encoders.items():
            if feat_idx < len(feature_names):
                feat_name = feature_names[feat_idx]
                if feat_name in nn_df.columns:
                    if feat_idx in feature_encoders:
                        try:
                            nn_df[feat_name] = encoder.inverse_transform(nn_df[feat_name].astype(int))
                        except:
                            pass
        
        # Replace values that are the same as original with '-'
        for i in range(len(nearest_neighbors)):
            for j in range(len(feature_names)):
                if nn_df.iloc[i, j] == original_df.iloc[0, j]:
                    nn_df.iloc[i, j] = '-'
        
        display(nn_df)
        
        counterfactuals = cf_result.counterfactual
        
        # Create a DataFrame for the counterfactuals
        cf_df = pd.DataFrame(counterfactuals, columns=feature_names)
        
        # Convert encoded categorical values back to original values for counterfactuals
        for feat_idx, encoder in feature_encoders.items():
            if feat_idx < len(feature_names):
                feat_name = feature_names[feat_idx]
                if feat_name in cf_df.columns:
                    if feat_idx in feature_encoders:
                        try:
                            cf_df[feat_name] = encoder.inverse_transform(cf_df[feat_name].astype(int))
                        except:
                            pass
        
        # Replace values that are the same as original with '-'
        for i in range(len(counterfactuals)):
            for j in range(len(feature_names)):
                if cf_df.iloc[i, j] == original_df.iloc[0, j]:
                    cf_df.iloc[i, j] = '-'
        
            
        # Display counterfactuals
        print("\nCounterfactual instances (only showing differences from original):")
        display(cf_df)
        

In [13]:
display_counterfactuals(results, feature_names, feature_encoders,  label_encoder)

Original instance predicted as: C

Original instance:


,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,D-penicillamine,14872.0,F,N,N,N,N,2.1,392.0,3.43,52.0,1395.0,184.45,194.0,328.0,10.2,3.0



----- Target Class: CL -----

Nearest Neighbors (only showing differences from original):


,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,-,13329.0,-,-,-,-,-,3.4,450.0,3.37,32.0,1408.0,116.25,118.0,313.0,11.2,2.0
1,-,12912.0,-,-,-,-,-,2.4,646.0,3.83,102.0,855.0,127.00,-,306.0,10.3,-
2,Placebo,14705.0,-,-,-,-,-,0.5,201.0,3.73,44.0,1345.0,54.25,145.0,445.0,10.1,2.0



Counterfactual instances (only showing differences from original):


,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,-,12912.0,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
1,-,14856.0,-,-,-,-,-,0.5,201.0,-,44.0,1363.0,54.25,-,445.0,10.14,2.0
2,-,-,-,-,-,-,-,2.28,404.6,-,-,1392.0,-,-,318.0,10.26,-



----- Target Class: D -----

Nearest Neighbors (only showing differences from original):


,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,-,13616.0,-,-,-,-,-,1.1,345.0,4.40,75.0,1860.0,218.55,72.0,447.0,10.7,-
1,Placebo,12963.0,-,-,-,-,-,1.3,408.0,4.22,67.0,1387.0,142.60,137.0,295.0,10.1,-
2,-,12227.0,-,-,-,-,-,1.6,660.0,4.22,94.0,1857.0,151.90,155.0,337.0,11.0,2.0



Counterfactual instances (only showing differences from original):


,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,-,12227.0,-,-,-,-,-,1.7000,660.0,4.220,94.0,1857.0,151.900,155.0,333.0,11.0,2.0
1,-,14869.0,-,-,-,-,-,2.0200,660.0,3.588,94.0,1857.0,183.148,185.0,333.0,10.52,2.0
2,-,-,-,-,-,-,-,1.9464,-,3.746,67.0,1392.0,142.600,176.0,-,-,-
